This notebook accompanies [13-symmetry-and-graphs](https://xing-phys.github.io/ai-for-physics/13-symmetry-and-graphs.html). Run cells from top to bottom. The web chapter provides typeset citations and numbered cross-references. Teaching code: MIT; text and figures: CC BY 4.0. All data are generated locally.


# Symmetry, geometry, and graph networks 

::: {.callout-note}
## The physical question
How can a learned model know that rotating a molecule changes its force vectors but does not change its energy?

This chapter develops group actions, invariance and equivariance, graph message passing, conservative forces, and geometric representations. You will construct and test a small learned potential, and learn which physical guarantees follow from symmetry and which require additional assumptions.
:::

## Choosing coordinates should not change the experiment

Imagine calculating the energy of an isolated water molecule. Relabel its two hydrogen atoms, translate all three atoms, or rotate your coordinate axes. No internal physical property should change. A generic network taking the nine Cartesian coordinates as inputs does not automatically respect any of these facts. It could assign a different energy after translation, simply because training happened near the origin.

One remedy is **data augmentation**: include rotated, translated, and relabeled copies in training. Another is to construct the architecture so that the desired transformation rule is an identity for every parameter value. This is an **inductive bias**, a restriction on possible learned functions motivated by the problem. Exact symmetry can reduce the amount of data spent teaching coordinate conventions and prevent certain unphysical errors outside the training set. It does not determine the correct interaction law.

Symmetry must match the actual experiment. A molecule in an external electric field has an orientation-dependent energy if the field is held fixed. Rotating both molecule and field preserves the scalar energy; rotating just the molecule need not. A crystal with fixed boundary conditions, a trapped atom, and a parity-violating observable require similarly careful choices. Enforcing an inapplicable symmetry can remove the physics one hopes to learn.

## Group actions from the beginning

A **group** is a set of transformations with an associative composition, an identity, and an inverse for each transformation. Three-dimensional rotations form the group $SO(3)$: real matrices $R$ satisfying $R^TR=I$ and $\det R=1$. Multiplying rotations composes them, $I$ does nothing, and $R^{-1}=R^T$. Including reflections gives $O(3)$, for which $\det R$ may be $-1$. Adding translations produces the Euclidean group $E(3)$. The subgroup containing rotations and translations but excluding reflections is $SE(3)$.

A **group action** tells us how a transformation acts on an object. A point transforms as $r\mapsto Rr+a$, while a displacement transforms as $r_i-r_j\mapsto R(r_i-r_j)$. The translation cancels for a displacement. A scalar such as energy is unchanged; a polar vector such as force transforms as $F\mapsto RF$. A map $f$ is **invariant** if $f(gx)=f(x)$ and **equivariant** if

$$
f(gx)=\rho(g)f(x),
$$ 

where $\rho(g)$ is the appropriate action on the output. Invariance is equivariance with the trivial output action. The central idea is that transforming first and predicting second must agree with predicting first and transforming the prediction.

For atom positions collected as an $N\times3$ array, a permutation matrix $P$ relabels the rows. An energy should obey $E(Pr,PZ)=E(r,Z)$ when species labels $Z$ are permuted consistently. Atomic forces should obey $F(Pr,PZ)=PF(r,Z)$. Permuting different species is a relabeling of the full position-species pairs, not a claim that replacing carbon by oxygen leaves the energy unchanged.

### Constructing invariants without group theory machinery

Distances are invariant because

$$
\|R(r_i-r_j)\|^2=(r_i-r_j)^TR^TR(r_i-r_j)=\|r_i-r_j\|^2.
$$

Dot products of displacement vectors are invariant by the same calculation. An angle can be represented by
$\cos\theta_{ijk}=\widehat r_{ij}\cdot\widehat r_{ik}$; using the cosine avoids the singular derivative of inverse cosine near collinear geometries. These quantities encode shape without choosing an orientation.

Distances alone do not distinguish mirror images when the full set of labeled pair distances is given: reflected coordinates have exactly the same distances. For reflection-invariant energies this is appropriate. To predict a pseudoscalar that changes sign under reflection, one can use an oriented triple product
$(r_{ij}\times r_{ik})\cdot r_{il}$. Its sign reverses under improper rotations. A model's features must therefore reflect the transformation type of its target, not a blanket preference for maximum invariance.

## Why physical systems are naturally graphs

A **graph** consists of nodes and edges. For a molecule, nodes can be atoms and edges interacting neighbors. Each node stores features, such as species and a learned vector of scalar numbers; each edge stores geometry or bond information. The graph may be fully connected, based on chemical bonds, or constructed from a distance cutoff. A mesh, a lattice Hamiltonian, an interacting particle cloud, and a detector event can all produce useful graphs, with different meanings assigned to their edges.

Graph models avoid treating the arbitrary position in an input list as a physical identity. A typical message-passing layer is

$$
m_i^{(\ell)}=\sum_{j\in\mathcal N(i)}
\phi_\theta^{(\ell)}(h_i^{(\ell)},h_j^{(\ell)},e_{ij}),
\qquad
h_i^{(\ell+1)}=\psi_\theta^{(\ell)}(h_i^{(\ell)},m_i^{(\ell)}).
$$ 

The functions $\phi$ and $\psi$ have shared parameters across nodes and edges. A **message** is simply an intermediate feature calculated from a neighboring pair. A node collects messages with a sum, then updates its own features. Under relabeling, the same physical neighbors appear in a different order. Addition is order independent, so $m_i$ and $h_i$ follow the node permutation. An invariant final energy can be obtained by summing node contributions:
$E_\theta=\sum_i\varepsilon_\theta(h_i^{(L)})$.

The proof has three ingredients: shared local functions, consistently relabeled edge features, and a symmetric aggregation operation. Concatenating neighbors in an arbitrary order would break it. Mean aggregation also preserves permutation symmetry but encodes a different dependence on coordination; summation naturally retains information about neighbor count. An attention-weighted sum can preserve symmetry if its weights are computed and normalized consistently over the neighbor set.

After one layer, a node knows about its direct neighbors; after $L$ layers, it can depend on nodes up to $L$ edges away. This region is the **receptive field**. Increasing the number of layers extends information propagation, but may compress many different environments into similar features or make optimization difficult. Long-range electrostatics cannot in general be recovered reliably from a fixed local cutoff just by fitting more short-range data.

The message-passing framework unified several molecular network constructions [@p-gilmer2017]. Particle-based graph simulators subsequently showed how messages can represent learned interactions and support long rollouts in several physical domains [@p-sanchez2020]. Their successes are numerical observations on specified systems; generic message passing alone does not impose Newton's third law, conservation, or rotational equivariance.

## From an invariant energy to equivariant forces

Suppose $E_\theta(r)$ is differentiable and invariant under rotations and translations. Define

$$
F_i(r)=-\frac{\partial E_\theta}{\partial r_i}.
$$ 

Let $r_i'=Rr_i+a$. A displacement obeys $\delta r_i'=R\delta r_i$. Because energy changes are coordinate independent,
$\sum_i(\nabla_{r_i'}E)\cdot R\delta r_i=\sum_i(\nabla_{r_i}E)\cdot\delta r_i$.
For every displacement this implies $R^T\nabla_{r_i'}E=\nabla_{r_i}E$, so $F_i(r')=RF_i(r)$. Scalar invariance has generated vector equivariance through differentiation.

Translation invariance gives more. Shift every position infinitesimally by $a$:

$$
0=\delta E=\sum_i\nabla_{r_i}E\cdot a
\quad\Longrightarrow\quad \sum_iF_i=0.
$$

For an infinitesimal rotation with axial vector $\delta\omega$, $\delta r_i=\delta\omega\times r_i$. Using the scalar triple product,
$0=-\delta\omega\cdot\sum_i r_i\times F_i$, hence the total torque vanishes. These are force-level consequences of symmetry. In a time-independent learned potential, exact continuous Newtonian evolution also conserves the learned mechanical energy:
$dE_{\mathrm{total}}/dt=\sum_i v_i\cdot(m_i\dot v_i+\nabla_iE)=0$.
A finite time integrator need not conserve it exactly.

This construction also gives a conservative force field, so work around a closed path is zero on a smooth single-valued potential domain. An arbitrary equivariant force network need not be conservative. Conversely, a conservative potential may have the wrong energies and barriers while satisfying every structural identity above. Physical consistency and numerical accuracy are different tests.

### Worked example: learning a radial interaction

Consider particles with pair potential
$V(r)=a r^2+b r^4$, where $r=\|r_i-r_j\|$. The total energy is a sum over unordered pairs. For one pair set $d=r_i-r_j$. The derivative of $r^2=d\cdot d$ with respect to $r_i$ is $2d$; the derivative of $r^4$ is $4r^2d$. Thus

$$
F_i^{(ij)}=-(2a+4br^2)d,\qquad F_j^{(ij)}=-F_i^{(ij)}.
$$

Even a linear fit for $a,b$ produces a model that is nonlinear in coordinates. The following demonstration learns from radial energy data and checks the geometric identities on an unrelated configuration. It is deliberately small enough that every operation is visible. A neural radial function would replace the polynomial, while the symmetry argument would remain the same.

In [1]:
import numpy as np

rng = np.random.default_rng(1301)
radius = np.linspace(0.4, 1.8, 30)
design = np.column_stack([radius**2, radius**4])
target = 0.7 * radius**2 + 0.2 * radius**4
a, b = np.linalg.lstsq(design, target, rcond=None)[0]

def energy_force(positions):
    energy = 0.0
    force = np.zeros_like(positions)
    for i in range(len(positions)):
        for j in range(i):
            d = positions[i] - positions[j]
            r2 = d @ d
            energy += a * r2 + b * r2**2
            fij = -(2 * a + 4 * b * r2) * d
            force[i] += fij
            force[j] -= fij
    return energy, force

positions = rng.normal(size=(4, 3))
rotation, _ = np.linalg.qr(rng.normal(size=(3, 3)))
rotation[:, 0] *= np.linalg.det(rotation)
E, F = energy_force(positions)
Er, Fr = energy_force(positions @ rotation.T + np.array([2., -1., 3.]))
assert np.allclose(E, Er)
assert np.allclose(Fr, F @ rotation.T)
assert np.allclose(F.sum(axis=0), 0)
assert np.allclose(np.cross(positions, F).sum(axis=0), 0)
perm = np.array([2, 0, 3, 1])
Ep, Fp = energy_force(positions[perm])
assert np.allclose(Ep, E) and np.allclose(Fp, F[perm])
print("Learned coefficients:", np.round([a, b], 8))

Learned coefficients: [0.7 0.2]


## Equivariant intermediate features

It is possible to keep every hidden feature invariant, using distances and angles as inputs. But geometric intermediate features can express interactions efficiently. Let $h_i$ be scalar node features and $v_i$ a vector. A vector message can be formed as

$$
v_i'=\sum_j\gamma_\theta(h_i,h_j,r_{ij}^2)(r_i-r_j).
$$

The coefficient $\gamma$ is invariant and each displacement rotates as a vector. Their weighted sum therefore rotates as a vector. Scalar messages can depend on $v_i\cdot v_j$, and vector updates may multiply vectors by learned scalars. Adding a fixed learned Cartesian vector would generally violate rotational equivariance, because a preferred direction would be built into the parameters.

An EGNN-style coordinate update uses this structure to change positions and scalar features while respecting Euclidean transformations [@p-satorras2021]. If pair coefficients are not symmetric under $i\leftrightarrow j$, the sum of position updates need not vanish. Equivariance alone therefore does not imply conservation of the center of mass. Even symmetric central updates must be interpreted with care if they are merely learned representation updates rather than physical accelerations.

For an extensive energy, the local atomic sum gives the correct duplication rule for two noninteracting copies, provided the graph has no edges between them. The individual atomic contributions are not uniquely observable: redistributing energy among atoms while preserving their sum can leave all total energies unchanged. Interpreting one learned atomic energy as an experimentally measured quantity is usually unjustified.

### Optional advanced section: angular channels

A three-dimensional scalar has one component and a vector has three. More general rotation representations contain $2\ell+1$ components labeled by angular momentum $\ell=0,1,2,\ldots$. Spherical harmonics $Y_{\ell m}(\widehat r)$ provide functions of direction that transform among themselves under a rotation:

$$
Y_{\ell m}(R\widehat r)=\sum_{m'=-\ell}^{\ell}
D^{(\ell)}_{mm'}(R)Y_{\ell m'}(\widehat r).
$$

The matrices $D^{(\ell)}$ encode the rotation action. This is the same mathematical structure that appears when rotating atomic orbitals. An equivariant network can carry several copies of these angular channels at each atom. Trainable linear maps mix copies of the same transformation type; nonlinear interactions combine channels through tensor products.

For a concrete first example, the product of two vectors contains a scalar $u\cdot v$, an antisymmetric part related to $u\times v$, and a symmetric traceless tensor
$T_{ab}=(u_av_b+u_bv_a)/2-\delta_{ab}(u\cdot v)/3$. These are respectively $\ell=0,1,2$ sectors under proper rotations. The tensor has five independent components: a symmetric $3\times3$ matrix has six, and imposing zero trace removes one. Clebsch–Gordan coefficients are the fixed numerical coefficients that perform this change of basis for general $\ell$. Their role is bookkeeping compatible with rotations, not additional learned physical laws.

Reflections require parity labels as well. A polar vector changes sign under inversion, whereas an axial vector such as angular momentum does not. Choosing $O(3)$ equivariance therefore requires tracking how channels behave under inversion, in addition to $SO(3)$ angular momentum. Readers can understand the basic network idea without memorizing the coefficients: combine geometric objects only through operations with a known transformation rule.

## Learning atomistic physics responsibly

A common training loss combines energy and force errors:

$$
\mathcal L=\sum_{c\in\mathcal D}\left[
w_E\frac{(E_\theta^{(c)}-E_{\mathrm{ref}}^{(c)})^2}{N_c}
+w_F\frac{1}{3N_c}\sum_i\|F_{\theta,i}^{(c)}-F_{\mathrm{ref},i}^{(c)}\|^2
\right].
$$

The weights carry the units needed to combine terms; normalization conventions differ among implementations. Force labels provide many derivative constraints per configuration and strongly constrain the shape of the energy surface. If forces are obtained by differentiating $E_\theta$, training on force errors requires mixed derivatives with respect to positions and model parameters. Smooth activation functions and smooth cutoff envelopes are important. An abrupt neighbor cutoff can make forces discontinuous when an atom crosses it.

Reference electronic-structure calculations are themselves approximations. A potential trained to a particular density functional reproduces that reference within its training domain; it does not automatically remove the functional's errors. Split data by trajectories, chemical composition, phases, or temperature ranges when those are the intended transfer tests. Randomly separating adjacent frames can greatly overestimate independent generalization because the geometries are nearly identical.

NequIP demonstrated the practical value of internal Euclidean equivariance for data-efficient interatomic potentials [@p-batzner2022]. MACE uses higher-body-order equivariant messages to obtain expressive local environments with relatively few message-passing layers [@p-batatia2022]. Here **body order** counts the atomic positions jointly entering a correlation term: a pair potential couples two atoms, while a bond-angle term couples three. Products of neighbor features can describe correlations involving several neighbors around a central atom. This is a property of the interaction representation, not merely the number of message-passing layers. Their architectural comparison involves accuracy, cost, body order, angular resolution, and the intended chemistry. No single benchmark ranking establishes a universally best potential.

The 2025 MACE-MP-0 study investigated a broadly pretrained atomistic model and application-specific fine tuning [@p-batatia2025]. The move from fitting one material to reusing a broad model changes the practical workflow, but broad chemical coverage is not an error certificate for every reaction, charge state, magnetic state, or extreme-pressure configuration. A sensible frontier problem is to connect geometric architectures to calibrated uncertainty and targeted reference calculations, especially for rare events that dominate transport or reaction rates.

Beyond atoms, graphs organize particles in fluids, sites in quantum spin systems, meshes in elasticity, and detector objects in high-energy physics. Euclidean symmetry may then be replaced or supplemented by lattice translations, gauge transformations, Lorentz transformations, or a permutation of identical subsystems. The reusable scientific procedure is: identify the actual transformation of inputs and outputs, build compatible features and operations, and test the resulting identities numerically.

## Implementations to inspect

The primary papers link the [NequIP implementation](https://github.com/mir-group/nequip) and [MACE implementation](https://github.com/ACEsuit/mace). Reading their energy and force calculation paths is a useful next step after the polynomial example. Software, pretrained weights, and training datasets can have different licenses and accuracy domains; record the specific artifacts used in a research calculation. Equivariance tests should accompany any custom architectural modification.

## Exercises and selected answers

1. **Energy and a field.** A dipole moment $\mu$ interacts with a uniform electric field $\mathcal E$ through $E=-\mu\cdot\mathcal E$. Show invariance under a common rotation and explain why rotating only $\mu$ is a different experiment.
2. **Permutation proof.** Express a scalar-feature graph layer as $H'=\sigma(AHW)$ on a fixed graph. Under a permutation $P$, use $A'=PAP^T$, $H'=PH$ to show that its output is permuted consistently. Rename intermediate arrays to avoid confusing the output with the transformed input.
3. **Conservation counterexample.** Consider $F_1=c(r_1-r_2)$ and $F_2=d(r_2-r_1)$ for scalar invariant coefficients $c\ne d$. Verify rotational and translation equivariance, and calculate the total force.
4. **Symmetry audit.** Add finite-difference force checks to the code. Then replace the energy by $E+\epsilon r_{1x}^2$. Which symmetry identities fail, and why?
5. **Locality.** Construct two widely separated charged clusters. Explain why a finite-range sum of local atomic energies misses their separation-dependent Coulomb interaction. Propose one explicit physical term and one learned long-range mechanism.

**Answer to 2.** Define $\Phi(A,H)=\sigma(AHW)$ with an elementwise nonlinearity. Then
$\Phi(PAP^T,PH)=\sigma(PAHW)=P\sigma(AHW)=P\Phi(A,H)$. The middle equality uses that a row permutation commutes with an elementwise operation.

**Answer to 3.** Displacements eliminate common translations and transform linearly under rotations, so both force rules are equivariant. Their sum is $(c-d)(r_1-r_2)$, which need not vanish. Equivariance constrains transformation behavior, while momentum conservation constrains relations between the forces.

**Answer to 5.** Once the clusters are beyond the cutoff, no local feature changes as their separation changes. The predicted interaction is consequently constant. An explicit charge model with a Coulomb sum, Ewald summation under periodic boundary conditions, or a learned global field can supply the missing dependence. Predicting the charges introduces additional physical and statistical assumptions.